In [9]:
%pip install azureml-fsspec==1.3.1


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml import command, Input, Output, MLClient
from azure.identity import DefaultAzureCredential
from azureml.fsspec import AzureMachineLearningFileSystem
import pandas as pd
import os 
import time
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes, InputOutputModes
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding, DataType

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [16]:
# 2. Define your target datastore and the relative folder path inside it
datastore_name = "ml_datastore1" 
relative_folder_path = "home_credit_card_risk/"

# 3. Construct the URI dynamically using the client configuration
datastore_uri = (
    f"azureml://subscriptions/{ml_client.subscription_id}/"
    f"resourcegroups/{ml_client.resource_group_name}/"
    f"workspaces/{ml_client.workspace_name}/"
    f"datastores/{datastore_name}/"
    f"paths/{relative_folder_path}"
)

In [4]:
import pandas as pd

df = pd.read_csv(os.path.join(datastore_uri,'HomeCredit_columns_description.csv'), encoding='latin1',index_col=0)
df.head()

,Table,Row,Description,Special
1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN


## LIST FILES IN DATASTORE

In [5]:
from azureml.fsspec import AzureMachineLearningFileSystem

# Instantiate the filesystem by using the following URI.
fs = AzureMachineLearningFileSystem(datastore_uri)

fs.ls() # List folders and files in datastore datastorename.

['home_credit_card_risk/HomeCredit_columns_description.csv',
 'home_credit_card_risk/POS_CASH_balance.csv',
 'home_credit_card_risk/application_test.csv',
 'home_credit_card_risk/application_train.csv',
 'home_credit_card_risk/bureau.csv',
 'home_credit_card_risk/bureau_balance.csv',
 'home_credit_card_risk/credit_card_balance.csv',
 'home_credit_card_risk/installments_payments.csv',
 'home_credit_card_risk/previous_application.csv',
 'home_credit_card_risk/sample_submission.csv']

## UPLOAD A FILE TO DATASTORE

In [7]:
# 2. Define your target datastore and the relative folder path inside it
datastore_name = "ml_datastore1" 
relative_folder_path = "data/"

# 3. Construct the URI dynamically using the client configuration
datastore_uri = (
    f"azureml://subscriptions/{ml_client.subscription_id}/"
    f"resourcegroups/{ml_client.resource_group_name}/"
    f"workspaces/{ml_client.workspace_name}/"
    f"datastores/{datastore_name}/"
    f"paths/{relative_folder_path}"
)

In [10]:
os.getcwd()

'/mnt/batch/tasks/shared/LS_root/mounts/clusters/mlops-ci001/code/Users/lishadalve/credit_risk_detection/data_administration'

In [11]:
fs = AzureMachineLearningFileSystem(datastore_uri)

# You can set recursive to False to upload a file.
fs.upload(lpath='/mnt/batch/tasks/shared/LS_root/mounts/clusters/mlops-ci001/code/Users/lishadalve/credit_risk_detection/data/ratings.csv', rpath='ratings', recursive=True, **{'overwrite': 'MERGE_WITH_OVERWRITE'})

# You need to set recursive to True to upload a folder.
# fs.upload(lpath='data/upload_folder/', rpath='data/fsspec_folder', recursive=True, **{'overwrite': 'MERGE_WITH_OVERWRITE'})

## READ DATA FROM ML TABLE

In [11]:
import mltable

# Define a path, folder, or pattern.
path = {
    'folder': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/mlops-ci001/code/Users/lishadalve/credit_risk_detection/ml_tables'
    # alternatives
    # 'file': '<supported_path>'
    # 'pattern': '<supported_path>'
}

# Create an mltable from paths.
tbl = mltable.load(uri = path['folder'])
# alternatives
# tbl = mltable.from_parquet_files(paths=[path])
# tbl = mltable.from_json_lines_files(paths=[path])
# tbl = mltable.from_delta_lake(paths=[path])

# Materialize to Pandas.
df = tbl.to_pandas_dataframe()
df.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [ ]:
#Reading directly fromm the asset

data_asset = ml_client.data.get(name="credit_data_description", version="1788954834")

tbl = mltable.load(f'azureml:/{data_asset.id}')
df = tbl.to_pandas_dataframe()
df.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [7]:
df.shape

(10001358, 8)

In [3]:
#Take a randome sample of the large dataset
path = {
    'file': 'azureml://subscriptions/5cc08b97-8906-4620-861c-088452556a7b/resourcegroups/demoGroup/workspaces/mlw-mlops-3nftht/datastores/ml_datastore1/paths/home_credit_card_risk/POS_CASH_balance.csv'
}

tbl = mltable.from_delimited_files(paths=[path])
# Take a random 30% sample of the data.
tbl = tbl.take_random_sample(probability=0.3)
df = tbl.to_pandas_dataframe()
df.head()

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1784872,397406,-32,12.0,9.0,Active,0,0
1,2371489,274851,-32,24.0,16.0,Active,0,0
2,2328294,287361,-32,12.0,12.0,Active,0,0
3,1846127,380050,-31,36.0,28.0,Active,0,0
4,1381005,195362,-34,60.0,56.0,Active,0,0


In [4]:
df.shape

(3000469, 8)

In [13]:
#Using fiter method  to get a suubset
filtered_table = tbl.filter('NAME_CONTRACT_STATUS == "Active"')
df_filtered = filtered_table.to_pandas_dataframe()
df_filtered.shape

(9151119, 8)

In [14]:
df_filtered.shape

(9151119, 8)